# RAG

In [ ]:
import time
start_time = time.time()

### Upload html documents from local folder

BSHTMLLoader: Strips all HTML immediately → tables become unformatted text  
Your custom function: Converts tables to markdown first → tables remain structured

In [1]:
# Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

# type(documents)     -> list
# type(documents[0])  -> langchain_core.documents.base.Document
# documents[0].metadata ->    {'source': '6k_filings\\CIK0000932782_0000932782-23-000006_pemex_fsx6kxq1-2023.htm',    'docid': 65456}

### Split LangChain document objects into chunks that are as well LangChain document objects

Considered using MarkdownHeaderTextSplitter because used markdowns to clarify tables. Still, better RecursiveCharacterTextSplitter

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,    #  adds where each chunk starts in the original document
    separators=["\n\n", "\n", ". ", " ", ""]  # Splits on paragraphs, then lines, then sentences, then words, then characters
)
chunks = text_splitter.split_documents(documents)
print(f'Split {len(documents)} filings (documents) into {len(chunks)} chunks.' )

# type(chunks)     -> list
# type(chunks[0])  -> langchain_core.documents.base.Document
# chunks[0].metadata    ->      {'source': '6k_filings\\CIK0000932782_0000932782-23-000006_pemex_fsx6kxq1-2023.htm',    'docid': 65456,    'start_index': 0}

Split 1010 filings (documents) into 63427 chunks.


### Batch Embeddings
#### Prepare batches

In [3]:
import json

def create_batch_jsonl(
    chunks, 
    output_dir="batch_files",
    max_lines_per_file=10000
):
    """
    Create JSONL files for OpenAI batch embeddings with custom IDs and file limits
    
    Args:
        chunks: List of LangChain Document objects
        output_dir: Directory to save batch files
        max_lines_per_file: Maximum number of tasks per JSONL file
    
    Returns:
        List of created file paths
    """
    Path(output_dir).mkdir(exist_ok=True)
    batch_files = []

    global_counter = 0
    
    # Split chunks into batches
    for batch_num in range(0, len(chunks), max_lines_per_file):
        batch_chunks = chunks[batch_num:batch_num + max_lines_per_file]
        output_file = f"{output_dir}/batch_for_embeddings_{batch_num // max_lines_per_file + 1}.jsonl"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for chunk in batch_chunks:
                # Create unique custom_id from metadata
                custom_id = f"{global_counter}_{chunk.metadata['docid']}_{chunk.metadata['start_index']}"
                global_counter += 1
                
                out_dict = {
                    "custom_id": custom_id,
                    "method": "POST",
                    "url": "/v1/embeddings",
                    "body": {
                        "model": "text-embedding-3-small",
                        "input": chunk.page_content
                    }
                }
                f.write(json.dumps(out_dict, ensure_ascii=False) + '\n')
        
        batch_files.append(output_file)
        print(f"Created {output_file} with {len(batch_chunks)} tasks")
    
    print(f"\nTotal: {len(batch_files)} batch file(s) created")
    return batch_files

# Create batch files
batch_files = create_batch_jsonl(chunks, max_lines_per_file=10000)

Created batch_files/batch_for_embeddings_1.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_2.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_3.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_4.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_5.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_6.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_7.jsonl with 3427 tasks

Total: 7 batch file(s) created


In [ ]:
# 63427 chunks, every chunk is a task

#### Keep a mapping. To make retrieval easier, save the chunk list

In [ ]:
import pickle
import json

# After creating chunks, save them
with open('chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

# Save as JSON with UTF-8 encoding
chunk_index = {}
for chunk in chunks:
    custom_id = f"{chunk.metadata['docid']}_{chunk.metadata['start_index']}"
    chunk_index[custom_id] = {
        "source": chunk.metadata['source'],
        "docid": chunk.metadata['docid'],
        "start_index": chunk.metadata['start_index'],
        "page_content": chunk.page_content
    }

with open('chunk_index.json', 'w', encoding='utf-8') as f:  
    json.dump(chunk_index, f, ensure_ascii=False, indent=2)

# # When loading later
# with open('chunk_index.json', 'r', encoding='utf-8') as f:
#     chunk_index = json.load(f)

### Upload input file

In [6]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [7]:
from openai import OpenAI

client = OpenAI()
files = client.files.list()

In [8]:
len(files.to_dict()['data'])

389

In [8]:
from glob import glob

batch_files = glob('./batch_files/batch_for_embeddings_*.jsonl')
batch_files

['./batch_files\\batch_for_embeddings_3.jsonl']

In [9]:
from tqdm import tqdm
client = OpenAI()

my_batch_files_ids = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_batch_files_ids.append(batch_input_file.id)
    print(batch_input_file)

100%|██████████| 1/1 [00:06<00:00,  6.37s/it]

FileObject(id='file-EUFzZD34J6rFakwTHuTA4u', bytes=9102641, created_at=1762987311, filename='batch_for_embeddings_3.jsonl', object='file', purpose='batch', status='processed', expires_at=1765579311, status_details=None)


In [ ]:
my_batch_files_ids

# ['file-GBkhk7D2oaihV3NkzksFPq',
#  'file-QKFidckavw6uWUvNvQrdRY',
#  'file-V2DsizfRkextGNn21wDeAc',
#  'file-2E7G14V8DQL4oYB5XhQK5M']

# ['file-JjVmW8EKFhX9G27fTpahbj',
#  'file-1mFDBukfmxzew1XjUHeXpJ',
#  'file-M2tKq7thibqMfJHAfug2jq']

# Finalizing: file-V2DsizfRkextGNn21wDeAc. batch_for_embeddings_3 not done yet, resent below
# ['file-EUFzZD34J6rFakwTHuTA4u']

['file-EUFzZD34J6rFakwTHuTA4u']

In [11]:
my_id = 'antonio_m_lancuentra'

In [12]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Content embeddings ({my_id}) {timestamp}"

for file_id in tqdm(my_batch_files_ids):
    client.batches.create(
            input_file_id = file_id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

100%|██████████| 1/1 [00:00<00:00,  1.86it/s]


In [ ]:
batch_description

# 'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19'
# 'Content embeddings (antonio_m_lancuentra) 2025-11-11 17:30:25'
# batch_for_embeddings_3 repeated, below:
# 'Content embeddings (antonio_m_lancuentra) 2025-11-12 17:42:22'

'Content embeddings (antonio_m_lancuentra) 2025-11-12 17:42:22'

### After launching batches, I want to check the status
If I turn the computer off. Then I need to run the cells below

In [2]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [3]:
from openai import OpenAI
client = OpenAI()

In [ ]:
client.files.list().to_dict()

In [ ]:
client.batches.list().to_dict()

In [16]:
batch_description =  'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19'

In [17]:
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id']}  
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

[{'batch_id': 'batch_69138f8036bc81908ecbf415a1b8829d',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-R2hwFEmsxEDMA4gBCXTpgw'},
 {'batch_id': 'batch_69138f7fc9b48190bf0c772d23a2e800',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19',
  'status': 'finalizing',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': None},
 {'batch_id': 'batch_69138f7f79588190be07aeabcf9f28a3',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-G3VncFghrHu4kSz5ARf7dd'},
 {'batch_id': 'batch_69138f7f189c8190a6ba7e2ab8243530',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19',
  'status': 'completed',
  '

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")